In [2]:
import os 
import matplotlib.pyplot as plt
import xml.etree.ElementTree as ET
import requests
from sentence_transformers import SentenceTransformer
import numpy as np  
import sys

In [3]:
import sys
from pathlib import Path

extr = next((str(p/"extr") for p in [Path.cwd()] + list(Path.cwd().parents) if (p/"extr").is_dir()), None)
if extr and extr not in sys.path:
    sys.path.insert(0, extr)
print("extr hinzugefügt zu sys.path:", extr)

from redner_extraction import extract_speeches, extract_all_speakers, get_average_sentiment
from support_func import chunksizing

extr hinzugefügt zu sys.path: c:\Users\brand\OneDrive\Desktop\AI_Selbststudium\ML_little\bundestag_reden\extr


Device set to use cpu


### Extraction of bundestag data  

In [4]:
# Füge Parent-Dir von `extr` zu sys.path hinzu, damit z.B. config.yaml gefunden werden kann
parent_dir = str(Path(extr).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)
print("Parent-Dir hinzugefügt zu sys.path:", parent_dir)

print(os.listdir(parent_dir))


import yaml
# Load YAML data
with open(parent_dir + "\\config.yaml", "r") as file:
    data = yaml.safe_load(file)  # safe_load prevents arbitrary code execution

# Modify data (optional)
url = data['data']["xml_file"]

Parent-Dir hinzugefügt zu sys.path: c:\Users\brand\OneDrive\Desktop\AI_Selbststudium\ML_little\bundestag_reden
['.git', '.gitignore', '.gradio', '.venv', '.vscode', 'config.yaml', 'environment.yml', 'extr', 'party_program', 'README.md', 'results', 'train', 'web_ui']


In [5]:
response = requests.get(url)

# XML in einen Tree parsen
tree = ET.ElementTree(ET.fromstring(response.content))

# Wurzel-Element abrufen
root = tree.getroot()

In [6]:
reden = {}
redner = []
# Beispiel-Ausgabe
for speech in extract_speeches(root):
    #print(f"Redner: {speech['name']}\nRede: {speech['text'][:200]}...\n")
    if speech['name'] not in redner:
        redner.append(speech['name'])
        reden[speech['name']] = speech['text']
    else:
        print("DOPPELT:", speech['name'])
        reden[speech['name']] += "\n Nächste Rede:" + speech['text']

DOPPELT: Emmi Zeulner
DOPPELT: Sören Pellmann
DOPPELT: Christian Moser
DOPPELT: Klaus Wiener
DOPPELT: Agnes Conrad


In [7]:
redner_party = []
all_speakers = extract_all_speakers(root)
for speaker in all_speakers:
    redner_party.append((speaker['vorname'] + " " + speaker['nachname'], speaker['fraktion']))

In [8]:
# Use a pipeline as a high-level helper
from transformers import pipeline

pipe = pipeline("summarization", model="Shahm/t5-small-german")



Device set to use cpu


In [9]:
# Load model directly
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
model_name = "Shahm/t5-small-german"
model_name = "Shahm/bart-german"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

In [10]:
from transformers import pipeline

# using pipeline API for summarization task
#summarization = pipeline("summarization", model=, tokenizer="t5-small", max_length=1100, min_length=700) # first model tried
summarization = pipeline("summarization", model=model, tokenizer=tokenizer)

# Just an example text in German
original_text = """

Sehr geehrte/r Frau/Herr [Name],

mit großem Interesse bewerbe ich mich als Bankkaufmann in Ihrem Haus. Ich möchte meine Stärke
 im kundenorientierten Arbeiten, meine Sorgfalt im Umgang mit finanziellen Themen sowie meine 
 Freude an Beratung und Vertrieb in Ihre Filiale/Ihr Team einbringen.
In meiner bisherigen Tätigkeit/ Ausbildung zum Bankkaufmann habe ich Privatkundinnen und -kunden 
im täglichen Bankgeschäft betreut – von Kontoführung und Zahlungsverkehr über Service- und 
Beratungsgespräche bis hin zur Terminvorbereitung für komplexere Finanzthemen. Dabei war mir wichtig, 
verständlich zu kommunizieren, Bedürfnisse sauber zu erfassen und Lösungen zuverlässig umzusetzen.
Besonders schätze ich die Kombination aus persönlicher Beratung, strukturierten Prozessen und 
verantwortungsvollem Arbeiten.
Ich arbeite gewissenhaft, diskret und belastbar – auch bei hohem Kundenaufkommen. 
Gleichzeitig bringe ich Teamgeist mit und lege Wert auf eine saubere Dokumentation 
sowie die Einhaltung regulatorischer Vorgaben. Den Umgang mit gängigen Bank-/Office-Systemen 
beherrsche ich sicher und arbeite mich schnell in neue Tools ein.
Gern überzeuge ich Sie in einem persönlichen Gespräch von meiner Motivation und Eignung. 
Ich freue mich über Ihre Rückmeldung.
Mit freundlichen Grüßen.
Max Mustermann
"""

print(len(original_text))

summary_text = summarization(original_text)[0]['summary_text']
print(summary_text)


from rouge_score import rouge_scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
scores = scorer.score(original_text, summary_text)
print("ROUGE Scores:", scores)

Device set to use cpu


1308
Ich arbeite gewissenhaft, diskret und belastbar – auch bei hohem Kundenaufkommen.
ROUGE Scores: {'rouge1': Score(precision=1.0, recall=0.052083333333333336, fmeasure=0.09900990099009901), 'rougeL': Score(precision=1.0, recall=0.052083333333333336, fmeasure=0.09900990099009901)}


In [12]:
for section in chunksizing(500, reden["Aaron Valent"]):
    summary_text = summarization(section)[0]['summary_text']
    #print(f"{len(summary_text)/len(section)} Summary: {summary_text}")
    print(f"Section length: {section}")
    print(f"Summary: {summary_text}")
    scores = scorer.score(section, summary_text)
    print("ROUGE Scores:", scores)
    print("\n")

Your max_length is set to 128, but your input_length is only 127. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=63)


Section length: Herr Präsident! Liebe Kolleginnen und Kollegen! Die Bundesregierung will Schiedsverfahren modernisieren; das ist ja erst einmal ganz gut. Es bedeutet hier vor allem digitale Verhandlungen. Doch Digitalisierung darf wie immer kein Selbstzweck sein; nein, Digitalisierung muss genau den Menschen dienen, die täglich mit ihr arbeiten müssen. Und genau hier hat das Gesetz leider ein, zwei Schwächen. Stellen Sie sich zum Beispiel vor, eine Zeugin wird per Video zugeschaltet.
Summary: Die Bundesregierung will Schiedsverfahren modernisieren. Das ist ja erst einmal ganz gut. Es bedeutet hier vor allem digitale Verhandlungen.
ROUGE Scores: {'rouge1': Score(precision=1.0, recall=0.2638888888888889, fmeasure=0.4175824175824176), 'rougeL': Score(precision=1.0, recall=0.2638888888888889, fmeasure=0.4175824175824176)}


Section length: Die Verbindung ist mal instabil, der Ton setzt aus, man vergisst, das Mikrofon stumm zu schalten, mehrere Personen reden gleichzeitig, irgendwo läuft ei

Your max_length is set to 128, but your input_length is only 84. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=42)


Section length: Plötzliche Lautstärkespitzen und Störgeräusche können das Gehör von Dolmetscherinnen und Dolmetschern nachhaltig beschädigen. Für uns ist deswegen ganz klar: Wer Verfahren digitalisieren will, muss sicherstellen, dass niemand benachteiligt wird. Wir brauchen verbindliche technische Standards, wirksamen Gesundheitsschutz und klare Regeln darüber, wann Onlineverhandlungen überhaupt notwendig und geeignet sind.
Summary: Wer Verfahren digitalisieren will, muss sicherstellen, dass niemand benachteiligt wird. Wir brauchen verbindliche technische Standards, wirksamen Gesundheitsschutz und klare Regeln darüber, wann Onlineverhandlungen überhaupt notwendig und geeignet sind.
ROUGE Scores: {'rouge1': Score(precision=1.0, recall=0.5178571428571429, fmeasure=0.6823529411764707), 'rougeL': Score(precision=1.0, recall=0.5178571428571429, fmeasure=0.6823529411764707)}


Section length: Denn eine schlechte Internetverbindung darf weder die Gesundheit der Übersetzer/-innen noch die Rech

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import torch, json

model_id = "z-dickson/BART_political_event_detection"
tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to("cuda" if torch.cuda.is_available() else "cpu")


text = "Following the devastating floods in Slovenia, our party calls for stronger climate resilience measures."
inputs = tok(text, return_tensors="pt").to(model.device)
outputs = model.generate(**inputs, max_new_tokens=128)
response = tok.decode(outputs[0], skip_special_tokens=True)
response_json = json.loads(response)
response_json

{
  "response_to_event": "Yes",
  "event_name": "Floods in Slovenia",
  "country": "Slovenia",
  "political_issue": "Climate adaptation policy"
}


c:\Users\brand\anaconda3\envs\job\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\brand\.cache\huggingface\hub\models--z-dickson--BART_political_event_detection. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


{'response_to_event': 'Yes',
 'event_name': 'Floods in Slovenia',
 'country': 'Slovenia',
 'political_issue': 'Climate adaptation policy'}

## Calcutation of the rouge score 


A description of the rouge_score is given on this website: https://spotintelligence.com/2024/08/12/rouge-metric-in-nlp/


The rouge score describes how similar the summary is to the original text, given as precision, recall and f1-score


In [ ]:
from rouge_score import rouge_scorer

# Initialisierung des Scorers 
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL', 'rougeLsum'], use_stemmer=True)



precisions_r1 = []
recalls_r1 = []
f1s_r1 = []



precisions_rL = []
recalls_rL = []
f1s_rL = []
length_ratios = []

for name, text in reden.items():
    summary_text = summarization(text)[0]['summary_text']
    scores = scorer.score(text, summary_text)
    print(f"Redner: {name}\nZusammenfassung: {summary_text} ")
    print(f"ratio of the summarized text to the original: {len(summary_text)/len(text)}")
    print("ROUGE Scores:", scores["rouge1"])
    # Rouge-1 Scores speichern
    precisions_r1.append(scores["rouge1"].precision)
    recalls_r1.append(scores["rouge1"].recall)
    f1s_r1.append(scores["rouge1"].fmeasure)
    # Rouge-L Scores speichern
    precisions_rL.append(scores["rougeL"].precision)
    recalls_rL.append(scores["rougeL"].recall)
    f1s_rL.append(scores["rougeL"].fmeasure)
    # Längenverhältnis speichern
    length_ratios.append(len(summary_text)/len(text))

Token indices sequence length is longer than the specified maximum sequence length for this model (958 > 512). Running this sequence through the model will result in indexing errors


Redner: Bärbel Bas
Zusammenfassung: Liebe Kolleginnen und Kollegen! 500 Milliarden Euro - noch nie hat eine Bundesregierung so viel Geld in die Hand genommen, um unser Land voranzubringen. Wir investieren in Kitas, in Schulen, in Krankenhäuser, in Klimaschutz. 
ratio of the summarized text to the original: 0.05705552725420275
ROUGE Scores: Score(precision=1.0, recall=0.05723905723905724, fmeasure=0.10828025477707007)
Redner: Hans-Jürgen Goßner
Zusammenfassung: Die Bundesregierung bastelt ein Gesetz, das angeblich für Fairness sorgen soll, in Wahrheit aber vor allem eins ist: ein Bürokratiemonster, das Betriebe gängelt, Unternehmer misstrauisch beäugt und Beamtenstellen sichert. 
ratio of the summarized text to the original: 0.04603207665069777
ROUGE Scores: Score(precision=1.0, recall=0.04423592493297587, fmeasure=0.0847240051347882)
Redner: Wilfried Oellers
Zusammenfassung: Sehr geehrte Frau Präsidentin! Liebe Kolleginnen und Kollegen! Meine sehr geehrten Damen und Herren! Heute berat